# 🎯 [TRN01] 드론 시뮬레이터 비행로그 기반 숙련도 분석기
### Google Colab 실습 환경 (교육·훈련 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/TRN01/trn01_m08_student.ipynb)

---

## 📌 1. 프로젝트 개요 및 학습 목표

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 교육·훈련 (Level 3) |
| **학습 소요 시간** | 10개 미션 (5일 / 총 27.5시간) |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 학습 목표 및 대상자
러시아-우크라이나 전쟁에서 소형 드론 조종 인력을 단기간에 양성·평가해야 했던 전장 요구를 배경으로, 교육용 가상 비행로그를 직접 생성·분석하며 드론 숙련도 평가 파이프라인 전 주기를 설계·구현하는 프로젝트다. 최종 목표는 교관 평가와 정합성을 갖춘 객관적 지표와 개인별 피드백을 산출하되, 등급과 재교육 결정은 교관이 근거를 기록해 확정하는 신뢰 가능한 분석기를 완성하는 것이다.

#### 본 프로젝트를 통해 익히는 핵심 역량

- **전술 시계열 데이터 설계 및 정제 역량:** 비행로그 스키마와 보안(가명화) 규칙을 정의하고, PWM 정규화·Hampel 필터·보간으로 센서 잡음과 단절을 복구하는 실무 기법을 익힌다.
- **도메인 기반 숙련도 모델링 및 한계 분석 역량:** 경로·호버링 Baseline 채점부터 장애물 회피·비상 착륙 반응 기반 정밀 평가 엔진까지 단계적으로 고도화하며, 혼동표·오탐 분석으로 모델의 사각지대를 찾아내는 방법을 습득한다.
- **통계적 신뢰도 검증 및 조작 습관 진단 역량:** 단발성 점수의 오류를 줄이기 위해 다회차 비행 묶음 평가와 부트스트랩 신뢰구간을 적용하고, 급조작·과보정 등 개인별 습관을 정량적으로 판별하는 원리를 익힌다.
- **HITL(인간-중심) 거버넌스 및 시스템 복원력 설계 역량:** 저신뢰·고위험 사례 우선 검토, 2단계 교관 확인, 해시 체인 감사 로그로 AI 책임 분계를 구현하고, 대량 배치 스트레스 시험과 AAR 재튜닝으로 운용 안정성을 확보하는 역량을 기른다.

#### 추천 학습 대상자 (페르소나)

- **드론·무인기 교육훈련 교관 및 평가 담당자:** 시뮬레이터 비행로그를 정량 지표로 해석하고, 교관의 경험적 평가와 AI 분석 결과를 결합해 개인별 맞춤 피드백·재교육 기준을 정립하고자 하는 현장 실무자.
- **국방 AI·데이터 분석 실무자 및 개발자:** Python 기본 문법을 바탕으로 시계열 노이즈 전처리, 다중 지표 채점 모델링, 부트스트랩 신뢰도 산출, HITL 승인 워크플로우까지 End-to-End 분석 파이프라인을 직접 구현하려는 개발자.
- **무인체계 시뮬레이터·훈련체계 기획 및 품질 검증 담당자:** 로그 스키마 표준화, 대량 배치 스트레스 테스트, 사후강평(AAR) 기반 파라미터 재튜닝, 감사 로그 거버넌스 등 신뢰 가능한 AI 보조 평가 체계의 설계 기준을 익히려는 엔지니어.


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 08. 100명 조종자 대량 비행로그 배치 분석 스트레스 테스트 및 성능 프로파일링
**소요시간**: 약 3.5시간 | **필수 산출물**: tests/trn01_stress_test.py (필수), 스트레스 테스트 분석 보고서 (필수)

### 📋 미션 개요 (Overview)
실제 교육훈련 환경에서는 다수 조종자의 로그가 한꺼번에 유입되거나 손상된 로그·배치 예외가 발생할 수 있어 시스템 복원력 검증이 필수적이다. 본 미션에서는 100~600건 규모의 부하, 손상 로그 5% 혼입, 묶음 처리 예외 상황을 주입하는 스트레스 테스트(`trn01_stress_test.py`)를 수행한다. 손상 로그 격리와 단건 축소 처리(Graceful Degradation)를 통해 로그 유실 0건·무중단 동작을 확인하고, 메모리 누수와 병목 함수 프로파일링 결과를 보고서로 정리한다.

### 🎓 학습 목표 (Objectives)
연구용 프로토타입을 넘어 **대량 부하와 비정상 입력·배치 장애 상황에서도 유실 없이 동작하는 시스템 복원력 검증 및 성능 프로파일링 역량**을 익힌다. 100~600건 부하, 손상 로그 5% 혼입, 묶음 처리 예외를 인위적으로 주입해 로그별 격리·단건 축소 처리(Graceful Degradation)를 구현하고, 처리량·메모리 누수·병목 함수를 계측해 개선안을 도출하는 실무 테스트 기법을 습득한다.

### 🛠️ 단계별 수행 및 AI 활용 가이드
#### Step 1. 미션8 코드를 실행해 상황별 채점·격리·유실·다운을 확인한다.
#### 이 단계의 역할
미션8 코드를 실행해 상황별 채점·격리·유실·다운을 확인한다.

#### 해야 할 일

- 미션8 코드를 실행해 상황별 채점·격리·유실·다운을 확인한다.에 필요한 데이터셋과 입력 파라미터를 준비한다.
- 정답 코드 및 모듈을 실행하여 중간 출력 결과를 도출한다.
- 출력된 지표와 로그가 기준 조건을 만족하는지 대조·검증한다.

#### 핵심 확인 지표
> 모든 상황 다운 0건, 로그 유실 0건(채점+격리=전체) 손상 로그 30건 전부 격리, 예외 시 기능 축소 1회

#### 학습자가 제출해야 하는 내용

- 미션8 코드를 실행해 상황별 채점·격리·유실·다운을 확인한다. 실행 결과 및 검증 로그

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> 100건·500건·600건 부하와 손상 로그 5% 혼입, 묶음 처리 예외 상황을 순차 주입해 채점 건수·격리 건수·유실 건수(전체 - 채점 - 격리)·다운 여부를 측정하는 스트레스 테스트를 작성해줘.
**AI 응답 검증 포인트**

- 모든 부하 상황에서 '채점 건수 + 격리 건수 == 전체 입력 건수(유실 0건)'가 성립하는지 확인
- 시스템 크래시(다운) 없이 전체 배치가 끝까지 완료되는지 실행 로그로 검증

#### Step 2. 격리 사유가 주입한 손상 유형과 맞는지 확인한다.
#### 이 단계의 역할
격리 사유가 주입한 손상 유형과 맞는지 확인한다.

#### 해야 할 일

- 격리 사유가 주입한 손상 유형과 맞는지 확인한다.에 필요한 데이터셋과 입력 파라미터를 준비한다.
- 정답 코드 및 모듈을 실행하여 중간 출력 결과를 도출한다.
- 출력된 지표와 로그가 기준 조건을 만족하는지 대조·검증한다.

#### 핵심 확인 지표
> 모든 상황 다운 0건, 로그 유실 0건(채점+격리=전체) 손상 로그 30건 전부 격리, 예외 시 기능 축소 1회

#### 학습자가 제출해야 하는 내용

- 격리 사유가 주입한 손상 유형과 맞는지 확인한다. 실행 결과 및 검증 로그

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> 격리된 손상 로그 목록을 출력해, 주입한 손상 유형(형식 오류, 결측 과다, 범위 이탈 등)과 격리 사유 코드가 정확히 매칭되는지 검증하는 리포트를 만들어줘.
**AI 응답 검증 포인트**

- 정상 로그가 실수로 격리(과탐)되거나 손상 로그가 채점 파이프라인으로 흘러들어가지 않았는지 확인
- 격리 사유 메시지만 보고도 운영자가 데이터 복구·재수집 조치를 할 수 있는지 점검

#### Step 3. 묶음 예외 상황의 처리량 감소와 병목 함수를 정리한다.
#### 이 단계의 역할
묶음 예외 상황의 처리량 감소와 병목 함수를 정리한다.

#### 해야 할 일

- 묶음 예외 상황의 처리량 감소와 병목 함수를 정리한다.에 필요한 데이터셋과 입력 파라미터를 준비한다.
- 정답 코드 및 모듈을 실행하여 중간 출력 결과를 도출한다.
- 출력된 지표와 로그가 기준 조건을 만족하는지 대조·검증한다.

#### 핵심 확인 지표
> 모든 상황 다운 0건, 로그 유실 0건(채점+격리=전체) 손상 로그 30건 전부 격리, 예외 시 기능 축소 1회

#### 학습자가 제출해야 하는 내용

- 묶음 예외 상황의 처리량 감소와 병목 함수를 정리한다. 실행 결과 및 검증 로그

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> 묶음 처리 예외 발생 시 단건 처리로 축소(Graceful Degradation)될 때의 처리량 변화와 cProfile 기반 병목 함수 Top 5를 추출해 개선안을 제시해줘.
**AI 응답 검증 포인트**

- 병목으로 지목된 함수명과 누적 실행 시간(cumtime)이 실제 프로파일러 출력과 일치하는지 확인
- 제안된 최적화(예: 캐싱, 벡터화)가 결과 정합성을 해치지 않는지 검토

#### Step 4. 도전 과제 및 파라미터 튜닝 최적화
#### 이 단계의 역할
기본 파이프라인 수행 후 심화 튜닝 및 조건 변경을 통해 최적화 역량을 다진다.

#### 해야 할 일
run_batch()의 chunk를 100에서 20으로 줄이면 처리량과 예외 시 손실 범위가 어떻게 달라지는지 비교한다.

#### 학습자가 제출해야 하는 내용

- 파라미터 변경 전후 비교 분석 메모 및 최종 최적화 결과

#### AI 활용 가이드 (프롬프트 예시 & 검증 포인트)
**AI 질의 예시**

> 동시 요청이나 반복 배치 실행 시 메모리 사용량 추이를 추적해 메모리 누수가 없는지 확인하고, 병목 함수 1개를 최적화해 처리량 개선 폭을 비교해줘.
**AI 응답 검증 포인트**

- tracemalloc 또는 psutil 기준 반복 실행 후 메모리가 무한 증가하지 않는지 수치로 확인
- 최적화 전후 동일 부하 테스트에서 초당 처리량(logs/sec)이 실제로 향상됐는지 검증


In [ ]:
"""조종자 100명의 대량 비행로그를 배치 분석하는 스트레스 테스트를 수행한다. 평시 100건에서 500% 증가(600건), 손상 로그 혼입, 묶음 처리 중 예외 상황을 주입해 시스템 다운 0건·로그 유실 0건(채점+격리=전체)과 처리량·메모리·병목 함수를 측정하고, 예외 시 1건씩 처리로 기능을 축소하는 안전 실패 동작을 확인한다."""
import cProfile
import io
import pstats
import time
import tracemalloc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge

# ---------------------------------------------------------------------------
# [교육용가상자료] 드론 조종 시뮬레이터 비행로그 생성기 (실제 훈련생·장비 정보 없음, 조종자는 가명 P001~)
# 비행 1회 = 120초(10Hz): 경로 비행(0~60초) -> 호버링(60~85초) -> 돌발 장애물 회피(85초) -> 비상 착륙(100초 지시)
# 조종자마다 숨은 숙련도(skill)와 조작 습관(급조작 jerky, 과보정 overcorr)이 있고, 교관 점수는 이를 반영해 만든다.
# ---------------------------------------------------------------------------
DT, STEPS = 0.1, 1200
T = np.arange(STEPS) * DT
COND = {"day": dict(wind=0.4, gust=0.0, gps=0.25, delay=0.0), "night": dict(wind=0.4, gust=0.0, gps=0.5, delay=0.15),
        "gust": dict(wind=1.2, gust=2.5, gps=0.25, delay=0.0)}
OBST_T, LAND_T = 85.0, 100.0


def make_pilots(n=40, seed=7):
    rng = np.random.default_rng(seed)
    skill = rng.beta(2.2, 2.0, n)
    return pd.DataFrame({"pilot": [f"P{i + 1:03d}" for i in range(n)], "skill": skill,
                         "jerky": rng.random(n) < 0.15 + 0.35 * (1 - skill), "overcorr": rng.random(n) < 0.15 + 0.35 * (1 - skill)})


def target_path(t):
    """사각형 경로(40x30m, 고도 20m)를 3m/s로 비행 -> 60초 지점에서 호버링 -> 착륙 지시 후 고도 0."""
    # [TODO : 학생 실습 8] 100명 조종자 대량 비행로그 배치 분석 스트레스 테스트 및 성능 프로파일링 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def fly(pilots, cond="day", seed=0, delay_extra=0.0, obstacle_side=None):
    """여러 조종자의 비행 1회를 동시에 모의한다(벡터화). 반환: 로그 배열 dict, 비행별 정답 정보 DataFrame."""
    # [TODO : 학생 실습 8] 100명 조종자 대량 비행로그 배치 분석 스트레스 테스트 및 성능 프로파일링 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def one(log, j):
    """여러 조종자 로그에서 j번째 조종자 1명의 로그(DataFrame)를 꺼낸다."""
    return pd.DataFrame({k: (v if v.ndim == 1 else v[:, j]) for k, v in log.items()})
# ---------------------------------------------------------------------------

# ---- 비행로그 특징 추출 공통 함수(여러 조종자 로그를 한 번에 계산) ----
from scipy.ndimage import uniform_filter1d

BASE_F = ["path_rmse", "hover_drift", "hover_std", "alt_err"]
ADV_F = BASE_F + ["obs_rt", "obs_lateral", "touch_vz", "att_exceed", "jerk", "overshoot", "n_risk"]
RULES = {"near_collision": ("obs_lateral", "<", 2.5), "hard_landing": ("touch_vz", ">", 2.0), "attitude_exceed": ("att_exceed", ">=", 10)}


def detect(df):
    """위험 조작 사건 탐지 규칙: 장애물 2.5초 후 회피 거리 2.5m 미만, 접지 속도 2.0m/s 초과, 자세 35도 초과 1초 이상."""
    out = {}
    for ev, (f, op, th) in RULES.items():
        out[ev] = df[f] < th if op == "<" else (df[f] > th if op == ">" else df[f] >= th)
    return pd.DataFrame(out)


def features(log):
    """경로 이탈도·호버링 안정성(기본) + 장애물 반응·착륙 속도·자세 한계·급조작·과보정(고도화) 특징."""
    X, Y, A = (uniform_filter1d(log[k], 9, axis=0) for k in ("x", "y", "alt"))       # GPS 잡음 완화(이동평균 0.9초)
    h = np.hypot(X - log["tx"], Y - log["ty"])
    cr, hv = slice(0, 600), slice(620, 850)
    k0, k1 = int(OBST_T / DT), int((OBST_T + 2.5) / DT)
    side = np.sign(log["ty"][k0 + 1] - log["ty"][k0 - 1])
    moved = (log["stick_roll"][k0:k0 + 40] * side) > 0.3
    rt = np.where(moved.any(0), moved.argmax(0) * DT, 4.0)
    vz = -np.gradient(A, DT, axis=0)
    land = np.argmax((A < 0.4) & (T[:, None] >= LAND_T), axis=0)
    touch = np.array([vz[max(i - 6, 0):i - 2, j].mean() if i > 6 else 3.0 for j, i in enumerate(land)])
    ds = np.abs(np.diff(np.stack([log["stick_roll"], log["stick_pitch"]]), axis=1)) / DT
    return pd.DataFrame({
        "path_rmse": np.sqrt((h[cr] ** 2).mean(0)), "hover_drift": h[hv].mean(0), "hover_std": np.hypot(X[hv].std(0), Y[hv].std(0)),
        "alt_err": np.abs(A[:850] - log["talt"][:850]).mean(0),
        "obs_rt": rt, "obs_lateral": np.abs(Y[k1] - Y[k0]), "touch_vz": touch,
        "att_exceed": (np.maximum(np.abs(log["roll_deg"]), np.abs(log["pitch_deg"])) > 35).sum(0),
        "jerk": np.percentile(ds, 95, axis=(0, 1)),
        "overshoot": np.clip(np.abs(Y[k0:k0 + 120] - Y[k0]).max(0) - 6.0, 0, None)}).pipe(
        lambda d: d.assign(n_risk=detect(d).sum(axis=1)))                                     # overshoot: 회피 후 목표 6m를 넘어선 거리(과보정)


def make_dataset(n_pilots=40, seed=7, conds=("day", "night", "gust"), flights=1):
    """조종자 n명 x 조건 x 반복 비행 -> 비행별 특징 + 정답(교관 점수·등급·위험 사건) 표."""
    pilots = make_pilots(n_pilots, seed)
    out = []
    for c in conds:
        for f in range(flights):
            log, tr = fly(pilots, c, seed=seed * 100 + list(COND).index(c) * 10 + f)
            out.append(pd.concat([tr.reset_index(drop=True), features(log)], axis=1).assign(flight=f))
    return pd.concat(out, ignore_index=True)


def cond_z(df, ref, cols):
    """조건별 기준 집단(ref)의 평균·표준편차로 표준화: 돌풍처럼 어려운 조건에서 받는 불이익을 줄인다."""
    z = df.copy()
    z[cols] = z[cols].astype(float)
    for c in df["cond"].unique():
        r = ref[ref["cond"] == c][cols]
        z.loc[df["cond"] == c, cols] = (df.loc[df["cond"] == c, cols] - r.mean()) / (r.std() + 1e-9)
    return z


def to_grade(score):
    return np.where(score >= 75, "A", np.where(score >= 55, "B", "C"))

KEYS = ["t", "x", "y", "alt", "stick_pitch", "stick_roll", "throttle", "roll_deg", "pitch_deg", "tx", "ty", "talt"]


def check_log(L):
    """로그 1건 검증. 문제가 있으면 사유 문자열, 없으면 None."""
    if any(k not in L for k in KEYS):
        return "필드 누락"
    if len(L["t"]) != STEPS:
        return f"길이 오류({len(L['t'])})"
    if np.isnan(L["x"]).mean() > 0.2:
        return "결측 20% 초과"
    if np.any(np.diff(L["t"]) <= 0):
        return "시각 역행"
    if np.abs(L["stick_roll"]).max() > 1.0001:
        return "스틱 범위 초과"
    return None


def score_chunk(logs, conds, eng, ref):
    stacked = {k: np.stack([L[k] for L in logs], axis=1) if k != "t" else T for k in KEYS}
    stacked = {k: (pd.DataFrame(v).interpolate(limit_direction="both").values if v.ndim == 2 else v) for k, v in stacked.items()}
    f = features(stacked).assign(cond=conds)
    return np.clip(eng.predict(cond_z(f, ref, ADV_F)[ADV_F]), 0, 100)


def run_batch(logs, conds, eng, ref, chunk=100, fail_chunk=None):
    scored, quarantine, degraded, down = {}, {}, 0, 0
    try:
        ok = []
        for i, L in enumerate(logs):
            why = check_log(L)
            (quarantine.__setitem__(i, why) if why else ok.append(i))
        for c0 in range(0, len(ok), chunk):
            idx = ok[c0:c0 + chunk]
            try:
                if fail_chunk is not None and c0 // chunk == fail_chunk:
                    raise MemoryError("모의 자원 부족")
                for i, s in zip(idx, score_chunk([logs[i] for i in idx], [conds[i] for i in idx], eng, ref)):
                    scored[i] = s
            except Exception:                                          # 기능 축소: 1건씩 처리, 그래도 실패하면 격리
                degraded += 1
                for i in idx:
                    try:
                        scored[i] = score_chunk([logs[i]], [conds[i]], eng, ref)[0]
                    except Exception as e:
                        quarantine[i] = f"처리 실패({type(e).__name__})"
    except Exception as e:
        down = 1; print("다운:", e)
    return scored, quarantine, degraded, down


# 엔진 준비(미션 4 방식)
train = make_dataset(40, seed=7)
eng = Ridge(alpha=1.0).fit(cond_z(train, train, ADV_F)[ADV_F], train["instructor"])

# [교육용가상자료] 조종자 100명 x 비행 6회 = 로그 600건 생성
pilots = make_pilots(100, seed=12)
LOGS, CONDS = [], []
for f in range(6):
    cond = list(COND)[f % 3]
    log, _ = fly(pilots, cond, seed=900 + f)
    for j in range(len(pilots)):
        LOGS.append({k: (v if v.ndim == 1 else v[:, j].copy()) for k, v in log.items() if k in KEYS}); CONDS.append(cond)
rng = np.random.default_rng(5)
BAD = []
for i in rng.choice(len(LOGS), 30, replace=False):                    # 손상 로그 5% 주입
    L = {k: v.copy() for k, v in LOGS[i].items()}
    kind = i % 4
    if kind == 0:
        del L["throttle"]
    elif kind == 1:
        L["x"][:400] = np.nan
    elif kind == 2:
        L = {k: v[:500] for k, v in L.items()}
    else:
        L["stick_roll"][100] = 7.0
    BAD.append((i, L))
CORRUPT = list(LOGS)
for i, L in BAD:
    CORRUPT[i] = L

CASES = [("평시 100건", LOGS[:100], CONDS[:100], None), ("100명 x 5회 500건", LOGS[:500], CONDS[:500], None),
         ("500% 증가 600건", LOGS, CONDS, None), ("600건 + 손상 5%", CORRUPT, CONDS, None),
         ("600건 + 손상 + 묶음 예외", CORRUPT, CONDS, 2)]
rows = []
for name, logs, conds, fail in CASES:
    t0 = time.perf_counter()
    sc, qu, dg, dn = run_batch(logs, conds, eng, train, fail_chunk=fail)
    dt = time.perf_counter() - t0
    rows.append({"상황": name, "로그": len(logs), "채점": len(sc), "격리": len(qu), "유실": len(logs) - len(sc) - len(qu),
                 "다운": dn, "기능축소 묶음": dg, "처리량(건/s)": round(len(logs) / dt), "소요(s)": round(dt, 2)})
R = pd.DataFrame(rows)
print(R.to_string(index=False))
print(f"\n시스템 다운 {R['다운'].sum()}건, 로그 유실 {R['유실'].sum()}건(채점+격리=전체), 최저 처리량 {R['처리량(건/s)'].min()}건/s")
_, qu, _, _ = run_batch(CORRUPT, CONDS, eng, train)
print("격리 사유:", pd.Series(list(qu.values())).value_counts().to_dict(), f"/ 주입 {len(BAD)}건")

tracemalloc.start()
mem = []
for k in range(3):
    run_batch(LOGS[:200], CONDS[:200], eng, train)
    mem.append(tracemalloc.get_traced_memory()[0] / 1e6)
peak = tracemalloc.get_traced_memory()[1] / 1e6
tracemalloc.stop()
print(f"200건 처리 최대 메모리 {peak:.0f} MB, 반복 후 잔류 {[round(m, 1) for m in mem]} MB -> {'누수 의심' if mem[-1] > mem[0] * 1.5 + 1 else '누수 징후 없음'}")

pr = cProfile.Profile(); pr.enable()
run_batch(LOGS[:300], CONDS[:300], eng, train)
pr.disable()
s = io.StringIO(); pstats.Stats(pr, stream=s).sort_stats("cumulative").print_stats(30)
top = [ln.split() for ln in s.getvalue().splitlines() if ln.strip()[:1].isdigit() and ".py" in ln and "m08" in ln.split()[-1]]
print("[누적 시간 상위 함수(이 파일)]")
for f in top[:4]:
    print(f"  {f[0]:>6}회  {f[3]}s  {f[-1].split(':')[-1]}")
print("개선 방향: 비행별 보간(DataFrame.interpolate)을 numpy 일괄 보간으로 바꾸고, 묶음 크기를 메모리에 맞춰 조정한다.")

plt.figure(figsize=(8, 4))
plt.bar([f"case{i + 1}" for i in range(len(R))], R["처리량(건/s)"], color=["tab:blue"] * 3 + ["tab:orange", "tab:red"])
plt.ylabel("logs per second"); plt.title("Batch analysis stress test throughput (synthetic)")
plt.tight_layout(); plt.show()

# TODO: 스트레스 테스트를 tests/trn01_stress_test.py 로, 분석 보고서를 docs/trn01_m08_stress_report.md 로 저장하세요.


<details>
<summary><b>💡 [참고] 미션 08 예상 실행 결과 (Output)</b></summary>

```text
               상황  로그  채점  격리  유실  다운  기능축소 묶음  처리량(건/s)  소요(s)
          평시 100건 100 100   0   0   0        0      1297   0.08
   100명 x 5회 500건 500 500   0   0   0        0      1290   0.39
     500% 증가 600건 600 600   0   0   0        0      1214   0.49
     600건 + 손상 5% 600 570  30   0   0        0      1258   0.48
600건 + 손상 + 묶음 예외 600 570  30   0   0        1       264   2.28

시스템 다운 0건, 로그 유실 0건(채점+격리=전체), 최저 처리량 264건/s
격리 사유: {'스틱 범위 초과': 10, '필드 누락': 8, '길이 오류(500)': 6, '결측 20% 초과': 6} / 주입 30건
200건 처리 최대 메모리 22 MB, 반복 후 잔류 [0.0, 0.0, 0.0] MB -> 누수 징후 없음
[누적 시간 상위 함수(이 파일)]
       1회  0.398s  193(run_batch)
       3회  0.376s  186(score_chunk)
       3회  0.171s  188(<dictcomp>)
       3회  0.088s  155(cond_z)
개선 방향: 비행별 보간(DataFrame.interpolate)을 numpy 일괄 보간으로 바꾸고, 묶음 크기를 메모리에 맞춰 조정한다.
```
</details>
